In [ ]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 36.4 MB/s eta 0:00:00


In [2]:
import os

print(os.path.exists("app.py"))
print(os.path.getsize("app.py"), "bytes")

True
68493 bytes


In [5]:
from pathlib import Path

path = Path("app.py")
lines = path.read_text(encoding="utf-8").splitlines()

print("First 5 lines of app.py:")
for i, line in enumerate(lines[:5], start=1):
    print(f"{i}: {line!r}")

First 5 lines of app.py:
1: '%%writefile app.py'
2: 'import os, re, html, sqlite3, jwt, bcrypt, datetime, time, secrets, smtplib, ssl'
3: 'import pandas as pd'
4: 'import streamlit as st'
5: 'import plotly.graph_objects as go'


In [6]:
from pathlib import Path

path = Path("app.py")
lines = path.read_text(encoding="utf-8").splitlines()

if lines and lines[0].strip() == "%%writefile app.py":
    path.write_text("\n".join(lines[1:]) + "\n", encoding="utf-8")

print("Removed the extra first line.")

Removed the extra first line.


In [7]:
import ast
from pathlib import Path

code = Path("app.py").read_text(encoding="utf-8")
ast.parse(code)

print("Success: app.py has no Python syntax errors.")

Success: app.py has no Python syntax errors.


In [10]:
from pathlib import Path

code = Path("app.py").read_text(encoding="utf-8")

for word in [
    "from google.colab import userdata",
    "userdata.get",
    "os.getenv",
    "NGROK_AUTHTOKEN",
    "EMAIL_PASSWORD",
    "EMAIL_ADDRESS",
    "ngrok"
]:
    print(f"{word}: {'Found' if word in code else 'Not found'}")

from google.colab import userdata: Not found
userdata.get: Not found
os.getenv: Not found
NGROK_AUTHTOKEN: Not found
EMAIL_PASSWORD: Not found
EMAIL_ADDRESS: Not found
ngrok: Not found


In [12]:
from google.colab import userdata
import os

os.environ["JWT_SECRET"] = userdata.get("JWT_SECRET")
os.environ["ADMIN_EMAIL"] = userdata.get("ADMIN_EMAIL")
os.environ["ADMIN_PASSWORD"] = userdata.get("ADMIN_PASSWORD")
os.environ["EMAIL_ADDRESS"] = userdata.get("EMAIL_ADDRESS")
os.environ["EMAIL_PASSWORD"] = userdata.get("EMAIL_PASSWORD")
os.environ["NGROK_AUTHTOKEN"] = userdata.get("NGROK_AUTHTOKEN")

print("Colab secrets loaded into the current session.")

Colab secrets loaded into the current session.


In [13]:
import os

os.environ["SMTP_HOST"] = "smtp.gmail.com"
os.environ["SMTP_PORT"] = "587"
os.environ["SMTP_USER"] = os.environ["EMAIL_ADDRESS"]
os.environ["SMTP_PASSWORD"] = os.environ["EMAIL_PASSWORD"]
os.environ["SMTP_FROM"] = os.environ["EMAIL_ADDRESS"]

print("Gmail SMTP configuration prepared.")

Gmail SMTP configuration prepared.


In [14]:
from pathlib import Path

code = Path("app.py").read_text(encoding="utf-8")

for term in ["SMTP_HOST", "SMTP_PORT", "SMTP_USER", "SMTP_PASSWORD", "SMTP_FROM",
             "EMAIL_ADDRESS", "EMAIL_PASSWORD", "smtplib", "send_message"]:
    print(f"{term}: {'Found' if term in code else 'Not found'}")

SMTP_HOST: Found
SMTP_PORT: Found
SMTP_USER: Found
SMTP_PASSWORD: Found
SMTP_FROM: Found
EMAIL_ADDRESS: Not found
EMAIL_PASSWORD: Not found
smtplib: Found
send_message: Found


In [15]:
import os

required_settings = [
    "JWT_SECRET",
    "ADMIN_EMAIL",
    "ADMIN_PASSWORD",
    "SMTP_HOST",
    "SMTP_PORT",
    "SMTP_USER",
    "SMTP_PASSWORD",
    "SMTP_FROM",
    "NGROK_AUTHTOKEN",
]

for setting in required_settings:
    value = os.environ.get(setting)
    print(f"{setting}: {'Loaded' if value else 'MISSING'}")

JWT_SECRET: Loaded
ADMIN_EMAIL: Loaded
ADMIN_PASSWORD: Loaded
SMTP_HOST: Loaded
SMTP_PORT: Loaded
SMTP_USER: Loaded
SMTP_PASSWORD: Loaded
SMTP_FROM: Loaded
NGROK_AUTHTOKEN: Loaded


In [16]:
import os

required = [
    "JWT_SECRET", "ADMIN_EMAIL", "ADMIN_PASSWORD",
    "SMTP_HOST", "SMTP_PORT", "SMTP_USER",
    "SMTP_PASSWORD", "SMTP_FROM", "NGROK_AUTHTOKEN"
]

missing = [key for key in required if not os.environ.get(key)]

if missing:
    print("Missing settings:", ", ".join(missing))
else:
    print("All required settings are loaded. Ready to start.")

All required settings are loaded. Ready to start.


In [17]:
import os
import subprocess
import time
from pyngrok import ngrok

# Stop any old Streamlit process
subprocess.run(["pkill", "-f", "streamlit run app.py"], capture_output=True)
time.sleep(2)

# Configure ngrok
ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])

# Start Streamlit in the background
log_file = open("/content/streamlit.log", "w")
process = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.address", "0.0.0.0",
     "--server.headless", "true"],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

# Wait for Streamlit to start
time.sleep(8)

# Check whether the process is still running
if process.poll() is not None:
    print("Streamlit did not start. Check the log in the next step.")
else:
    public_url = ngrok.connect(8501, "http").public_url
    print("Your Authentication Portal link:")
    print(public_url)

Your Authentication Portal link:
https://boots-coastland-jovial.ngrok-free.dev


In [18]:
from pathlib import Path
import subprocess

print("=== Streamlit Log ===")
log_path = Path("/content/streamlit.log")

if log_path.exists():
    print(log_path.read_text(encoding="utf-8", errors="replace")[-5000:])
else:
    print("Log file not found.")

print("\n=== Running Streamlit Processes ===")
result = subprocess.run(
    ["pgrep", "-af", "streamlit"],
    capture_output=True,
    text=True
)
print(result.stdout if result.stdout else "No Streamlit process found.")

=== Streamlit Log ===


2026-10-22 06:14:23.252 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://8.235.50.62:8501

2026-10-22 06:14:42.303 157.35.111.164:0 - "GET /static/js/WidgetLabel.DZjDVnAx.js HTTP/1.1" 200
2026-10-22 06:14:42.307 157.35.111.164:0 - "GET /static/js/useBasicWidgetState.QdjeajhU.js HTTP/1.1" 200
2026-10-22 06:14:42.328 157.35.111.164:0 - "GET /static/js/WidgetLabelHelpIcon.BgW9s-G5.js HTTP/1.1" 200
2026-10-22 06:14:42.328 157.35.111.164:0 - "GET /static/js/TextInput.BxDbpba5.js HTTP/1.1" 200


=== Running Streamlit Processes ===
10269 /usr/bin/python3 /usr/local/bin/streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true



In [19]:
from pyngrok import ngrok

tunnels = ngrok.get_tunnels()

if tunnels:
    for tunnel in tunnels:
        print("Public app link:", tunnel.public_url)
else:
    print("No ngrok tunnel found. Run the ngrok connection cell again.")

Public app link: https://boots-coastland-jovial.ngrok-free.dev


In [20]:
from google.colab import userdata
import os

os.environ["JWT_SECRET"] = userdata.get("JWT_SECRET")
os.environ["ADMIN_EMAIL"] = userdata.get("ADMIN_EMAIL")
os.environ["ADMIN_PASSWORD"] = userdata.get("ADMIN_PASSWORD")
os.environ["EMAIL_ADDRESS"] = userdata.get("EMAIL_ADDRESS")
os.environ["EMAIL_PASSWORD"] = userdata.get("EMAIL_PASSWORD")
os.environ["NGROK_AUTHTOKEN"] = userdata.get("NGROK_AUTHTOKEN")

In [21]:
os.environ["JWT_SECRET"] = userdata.get("JWT_SECRET")
os.environ["ADMIN_PASSWORD"] = userdata.get("ADMIN_PASSWORD")
os.environ["EMAIL_PASSWORD"] = userdata.get("EMAIL_PASSWORD")
os.environ["NGROK_AUTHTOKEN"] = userdata.get("NGROK_AUTHTOKEN")